In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, max_error, r2_score

In [2]:
# Read csv file to parse data into a pd DF
dataDF = pd.read_csv('SNRE_Data_w_Dummy_for_SPSS.csv')

# Separate Input variables (x) and target variable (y)
x = dataDF.drop(['Unit Price ($ psf)','Floor Level','First Floor','Type of Sale', 'New Sale',
                 'Central Region','Region'], axis=1)
y = dataDF['Unit Price ($ psf)']

In [3]:
input_train, input_test, target_train, target_test = train_test_split(x, y, 
                                                                      test_size = 0.2,
                                                                      random_state = 42,
                                                                      shuffle = True)

In [4]:
# Split the dummy variable & numerical variable in Training set 
train_dummy = input_train.iloc[:,0:7]
train_var = input_train.iloc[:,7:]
train_index =  train_dummy.index #Retain the index for merging later on

# Split the dummy variable & numerical variable in Testing set
test_dummy = input_test.iloc[:,0:7]
test_var = input_test.iloc[:,7:]
test_index = test_dummy.index #Retain the index for merging later on

display(train_var.head(3))
display(train_dummy.head(3))
display(train_index)

display(test_var.head(3))
display(test_dummy.head(3))
display(test_index)

# print(input_train.info())
# print(train_dummy.info())
# print(train_var.info())

,Remaining Tenure (Years),Total Population,SORA (%),Real GDP (in $Sm),Total Multiple-User Factory Space (m2),US Dollar (Singapore Dollar Per US Dollar),TPI % Change,lagged psf,Rental Index % change,minDistToMRT
5125,50,5685807.0,0.10,462647.93,1634000.0,1.38,2.8,549.58,-12.1,447.66
2907,40,5703569.0,1.90,481335.21,1711000.0,1.36,-0.1,165.77,-6.1,1704.83
5349,50,5685807.0,0.18,462647.93,1651000.0,1.38,2.8,386.38,-12.0,447.66


,Non-First Floor,Resale,Subsale,East Region,North Region,North-East Region,West Region
5125,1,1,0,1,0,0,0
2907,1,1,0,0,1,0,0
5349,1,1,0,1,0,0,0


Int64Index([5125, 2907, 5349,  108,  683, 2337, 8861,  997, 5271, 8884,
            ...
            8322, 5578, 4426,  466, 6265, 5734, 5191, 5390,  860, 7270],
           dtype='int64', length=7437)

,Remaining Tenure (Years),Total Population,SORA (%),Real GDP (in $Sm),Total Multiple-User Factory Space (m2),US Dollar (Singapore Dollar Per US Dollar),TPI % Change,lagged psf,Rental Index % change,minDistToMRT
5151,29,5685807.0,0.27,462647.93,1754000.0,1.38,2.8,518.91,-23.3,2150.63
1112,29,5469724.0,0.05,411286.49,2268000.0,1.27,6.8,329.27,-3.6,1351.79
6398,22,5453566.0,0.07,503862.12,1561000.0,1.34,17.1,175.78,-22.9,2454.94


,Non-First Floor,Resale,Subsale,East Region,North Region,North-East Region,West Region
5151,1,0,0,0,0,0,1
1112,1,0,0,0,1,0,0
6398,1,1,0,0,0,0,1


Int64Index([5151, 1112, 6398, 4060, 5612, 3312, 1897, 8757, 2310, 1071,
            ...
             274, 1345, 7285, 7166, 3518, 8729, 4245, 7649, 7180, 7488],
           dtype='int64', length=1860)

In [5]:
#Adding Polynomial Features to numeric training set, up to a degree of 2
arr_train_var = np.asanyarray(train_var)
polyDeg = 2
poly = PolynomialFeatures(degree = polyDeg, include_bias = False)
train_var_poly = poly.fit_transform(arr_train_var)
train_var_poly

array([[ 5.00000000e+01,  5.68580700e+06,  1.00000000e-01, ...,
         1.46410000e+02, -5.41668600e+03,  2.00399476e+05],
       [ 4.00000000e+01,  5.70356900e+06,  1.90000000e+00, ...,
         3.72100000e+01, -1.03994630e+04,  2.90644533e+06],
       [ 5.00000000e+01,  5.68580700e+06,  1.80000000e-01, ...,
         1.44000000e+02, -5.37192000e+03,  2.00399476e+05],
       ...,
       [ 3.90000000e+01,  5.63867600e+06,  5.90000000e-01, ...,
         6.40000000e+01, -4.17400000e+03,  2.72223062e+05],
       [ 2.80000000e+01,  5.46972400e+06,  1.60000000e-01, ...,
         1.69000000e+00, -1.63105800e+03,  1.57417172e+06],
       [ 5.40000000e+01,  5.63867600e+06,  1.12000000e+00, ...,
         6.56100000e+01, -5.67170100e+03,  4.90294044e+05]])

In [6]:
# Convert the array of polynomials to a DF
train_polyDF = pd.DataFrame(data = train_var_poly,
                      index = train_index,
                      columns = ['x'+str(i+1) for i in range(train_var_poly.shape[1])])

train_polyDF.head(5)

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x56,x57,x58,x59,x60,x61,x62,x63,x64,x65
5125,50.0,5685807.0,0.10,462647.93,1634000.0,1.38,2.8,549.58,-12.1,447.66,...,7.84,1538.824,-33.88,1253.448,302038.1764,-6649.918,246024.9828,146.41,-5416.686,2.003995e+05
2907,40.0,5703569.0,1.90,481335.21,1711000.0,1.36,-0.1,165.77,-6.1,1704.83,...,0.01,-16.577,0.61,-170.483,27479.6929,-1011.197,282609.6691,37.21,-10399.463,2.906445e+06
5349,50.0,5685807.0,0.18,462647.93,1651000.0,1.38,2.8,386.38,-12.0,447.66,...,7.84,1081.864,-33.60,1253.448,149289.5044,-4636.560,172966.8708,144.00,-5371.920,2.003995e+05
108,27.0,5469724.0,0.12,411286.49,1938000.0,1.27,6.8,447.97,-0.5,423.07,...,46.24,3046.196,-3.40,2876.876,200677.1209,-223.985,189522.6679,0.25,-211.535,1.789882e+05
683,34.0,5637022.0,0.68,521934.96,992000.0,1.38,30.7,539.85,-7.0,308.52,...,942.49,16573.395,-214.90,9471.564,291438.0225,-3778.950,166554.5220,49.00,-2159.640,9.518459e+04


In [7]:
# Merge Polynomials with dummy variables - train_dummy with polyDF
display(train_dummy)
poly_TrainDF = train_dummy.merge(train_polyDF, left_index = True, right_index = True)
display(poly_TrainDF.info())


,Non-First Floor,Resale,Subsale,East Region,North Region,North-East Region,West Region
5125,1,1,0,1,0,0,0
2907,1,1,0,0,1,0,0
5349,1,1,0,1,0,0,0
108,1,1,0,0,0,0,1
683,1,1,0,1,0,0,0
...,...,...,...,...,...,...,...
5734,1,1,0,0,1,0,0
5191,1,1,0,0,0,0,0
5390,1,1,0,0,0,0,0
860,1,0,0,0,1,0,0


<class 'pandas.core.frame.DataFrame'>
Int64Index: 7437 entries, 5125 to 7270
Data columns (total 72 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Non-First Floor    7437 non-null   int64  
 1   Resale             7437 non-null   int64  
 2   Subsale            7437 non-null   int64  
 3   East Region        7437 non-null   int64  
 4   North Region       7437 non-null   int64  
 5   North-East Region  7437 non-null   int64  
 6   West Region        7437 non-null   int64  
 7   x1                 7437 non-null   float64
 8   x2                 7437 non-null   float64
 9   x3                 7437 non-null   float64
 10  x4                 7437 non-null   float64
 11  x5                 7437 non-null   float64
 12  x6                 7437 non-null   float64
 13  x7                 7437 non-null   float64
 14  x8                 7437 non-null   float64
 15  x9                 7437 non-null   float64
 16  x10                74

None

In [8]:
#Adding Polynomial Features to numeric testing set, up to a degree of 2
arr_test_var = np.asanyarray(test_var)
poly = PolynomialFeatures(degree = polyDeg, include_bias = False)
test_var_poly = poly.fit_transform(arr_test_var)
test_var_poly.shape

(1860, 65)

In [9]:
# Convert the array of polynomials to a DF
test_polyDF = pd.DataFrame(data = test_var_poly,
                           index = test_index,
                           columns = ['x'+str(i+1) for i in range(test_var_poly.shape[1])])

test_polyDF.head(5)

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,...,x56,x57,x58,x59,x60,x61,x62,x63,x64,x65
5151,29.0,5685807.0,0.27,462647.93,1754000.0,1.38,2.8,518.91,-23.3,2150.63,...,7.84,1452.948,-65.24,6021.764,269267.5881,-12090.603,1.115983e+06,542.89,-50109.679,4.625209e+06
1112,29.0,5469724.0,0.05,411286.49,2268000.0,1.27,6.8,329.27,-3.6,1351.79,...,46.24,2239.036,-24.48,9192.172,108418.7329,-1185.372,4.451039e+05,12.96,-4866.444,1.827336e+06
6398,22.0,5453566.0,0.07,503862.12,1561000.0,1.34,17.1,175.78,-22.9,2454.94,...,292.41,3005.838,-391.59,41979.474,30898.6084,-4025.362,4.315294e+05,524.41,-56218.126,6.026730e+06
4060,53.0,5469724.0,0.12,411286.49,1938000.0,1.27,6.8,563.61,4.9,989.13,...,46.24,3832.548,33.32,6726.084,317656.2321,2761.689,5.574836e+05,24.01,4846.737,9.783782e+05
5612,30.0,5469724.0,0.08,411286.49,2103000.0,1.27,6.8,326.99,2.3,879.74,...,46.24,2223.532,15.64,5982.232,106922.4601,752.077,2.876662e+05,5.29,2023.402,7.739425e+05


In [10]:
# Merge Polynomials with dummy variables - test_dummy with polyDF
display(test_dummy)
poly_TestDF = test_dummy.merge(test_polyDF, left_index = True, right_index = True)
display(poly_TestDF.info())

,Non-First Floor,Resale,Subsale,East Region,North Region,North-East Region,West Region
5151,1,0,0,0,0,0,1
1112,1,0,0,0,1,0,0
6398,1,1,0,0,0,0,1
4060,1,1,0,0,0,1,0
5612,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...
8729,1,1,0,0,1,0,0
4245,1,1,0,0,0,0,0
7649,1,1,0,0,1,0,0
7180,1,0,1,0,0,0,0


<class 'pandas.core.frame.DataFrame'>
Int64Index: 1860 entries, 5151 to 7488
Data columns (total 72 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Non-First Floor    1860 non-null   int64  
 1   Resale             1860 non-null   int64  
 2   Subsale            1860 non-null   int64  
 3   East Region        1860 non-null   int64  
 4   North Region       1860 non-null   int64  
 5   North-East Region  1860 non-null   int64  
 6   West Region        1860 non-null   int64  
 7   x1                 1860 non-null   float64
 8   x2                 1860 non-null   float64
 9   x3                 1860 non-null   float64
 10  x4                 1860 non-null   float64
 11  x5                 1860 non-null   float64
 12  x6                 1860 non-null   float64
 13  x7                 1860 non-null   float64
 14  x8                 1860 non-null   float64
 15  x9                 1860 non-null   float64
 16  x10                18

None

In [11]:
# Creates an LinearRegression Object
lrModel = LinearRegression()
# Trains an LR model with the training data - poly_TrainDF
lrModel.fit(poly_TrainDF, target_train)

lrModel_coef = [(x, y) for x, y in zip(lrModel.feature_names_in_, lrModel.coef_.round(5))]
lrModel_coef.append(('Intercept', lrModel.intercept_.round(2)))
display(lrModel_coef)

# Do prediction with lrModel with input_test
lr_pred = lrModel.predict(poly_TestDF)
lr_score = lrModel.score(poly_TestDF, target_test) # R2 values - Goodness of Fit

# Compute Error metrics for Baseline LR Model
lrpred_mae = mean_absolute_error(target_test,lr_pred)
lrpred_mse = mean_squared_error(target_test,lr_pred)
lrpred_mape = mean_absolute_percentage_error(target_test,lr_pred)
lrpred_max = max_error(target_test,lr_pred)

print('Mae: {}'.format(lrpred_mae.round(2)))
print('Mse: {}'.format(lrpred_mse.round(2)))
print('Mape: {}'.format(lrpred_mape.round(2)))
print('Max: {}'.format(lrpred_max.round(2)))
print('R2 Score: {}'.format(lr_score.round(2)))

[('Non-First Floor', -52.2796),
 ('Resale', -13.53719),
 ('Subsale', 18.38812),
 ('East Region', -43.44911),
 ('North Region', -42.59866),
 ('North-East Region', -20.03038),
 ('West Region', -27.37278),
 ('x1', 5.6435),
 ('x2', 0.10695),
 ('x3', 49.21525),
 ('x4', -0.11512),
 ('x5', -0.00105),
 ('x6', -0.40186),
 ('x7', 0.82167),
 ('x8', 0.99519),
 ('x9', 16.09351),
 ('x10', -0.18356),
 ('x11', -0.05469),
 ('x12', 0.0),
 ('x13', -0.13535),
 ('x14', 0.0),
 ('x15', -0.0),
 ('x16', -5.14917),
 ('x17', -0.02347),
 ('x18', 0.00635),
 ('x19', 0.04733),
 ('x20', -0.00016),
 ('x21', -0.0),
 ('x22', 6e-05),
 ('x23', -0.0),
 ('x24', 0.0),
 ('x25', -0.00758),
 ('x26', 1e-05),
 ('x27', -0.0),
 ('x28', -0.0),
 ('x29', 0.0),
 ('x30', -2.37535),
 ('x31', -0.00015),
 ('x32', -2e-05),
 ('x33', -230.60857),
 ('x34', 0.33883),
 ('x35', 0.02101),
 ('x36', 0.14506),
 ('x37', -0.00135),
 ('x38', 0.0),
 ('x39', -0.0),
 ('x40', 0.09825),
 ('x41', -0.00012),
 ('x42', -0.0),
 ('x43', -1e-05),
 ('x44', 0.0),
 ('

Mae: 44.71
Mse: 3945.32
Mape: 0.13
Max: 436.65
R2 Score: 0.78


In [12]:
from sklearn.model_selection import cross_validate
training_error=[]
cross_validation_error=[]

# Make predictions with Training Data
training_pred = lrModel.predict(poly_TrainDF)

# Find Error between training pred vs actual y
mae_train = mean_absolute_error(target_train,training_pred)

cve = cross_validate(lrModel,poly_TrainDF,target_train,
                     scoring = ['neg_mean_absolute_error','r2'],
                     cv = 10)


display(cve)
display(np.mean(np.absolute(cve['test_neg_mean_absolute_error'])))


{'fit_time': array([0.01752067, 0.01350832, 0.01562905, 0.01423526, 0.01277876,
        0.01315928, 0.01385736, 0.01402497, 0.01381707, 0.01411343]),
 'score_time': array([0.00400281, 0.00452375, 0.00354743, 0.0034883 , 0.00384688,
        0.00350475, 0.00300074, 0.00300288, 0.00303888, 0.00234103]),
 'test_neg_mean_absolute_error': array([-44.77262547, -42.9317401 , -43.97890801, -46.44469676,
        -45.95381081, -42.54056457, -41.89541208, -44.18695177,
        -44.6706646 , -42.08716354]),
 'test_r2': array([0.74245285, 0.77526316, 0.78034538, 0.74048005, 0.72250646,
        0.78145672, 0.80132315, 0.77203725, 0.76307495, 0.79955368])}

43.94625377149708

In [13]:
# termList = []
# for t in lrModel_coef:
#     if t[1] == 0.0:
#         termList.append(t[0])
# print(termList)

In [14]:
# # remove useless regressors
# poly_TrainDF_parsi =  poly_TrainDF.drop(columns = termList, axis=1)
# poly_TestDF_parsi = poly_TestDF.drop(columns = termList, axis=1)

# display(poly_TrainDF_parsi.info())
# display(poly_TestDF_parsi.info())

In [15]:
# # Creates an LinearRegression Object
# lrModel_parsi = LinearRegression()
# # Trains an LR model with the training data - poly_TrainDF
# lrModel_parsi.fit(poly_TrainDF_parsi, target_train)

# lrModel_parsi_coef = [(x, y) for x, y in zip(lrModel_parsi.feature_names_in_, lrModel_parsi.coef_.round(5))]
# lrModel_parsi_coef.append(('Intercept', lrModel_parsi.intercept_.round(2)))
# display(lrModel_parsi_coef)

# # Do prediction with lrModel with input_test
# lr_parsi_pred = lrModel_parsi.predict(poly_TestDF_parsi)
# lr_parsi_score = lrModel_parsi.score(poly_TestDF_parsi, target_test) # R2 values - Goodness of Fit

# # Compute Error metrics for Baseline LR Model
# lrpred_parsi_mae = mean_absolute_error(target_test,lr_parsi_pred)
# lrpred_parsi_mse = mean_squared_error(target_test,lr_parsi_pred)
# lrpred_parsi_mape = mean_absolute_percentage_error(target_test,lr_parsi_pred)
# lrpred_parsi_max = max_error(target_test,lr_parsi_pred)

# print('Mae: {}'.format(lrpred_parsi_mae.round(2)))
# print('Mse: {}'.format(lrpred_parsi_mse.round(2)))
# print('Mape: {}'.format(lrpred_parsi_mape.round(2)))
# print('Max: {}'.format(lrpred_parsi_max.round(2)))
# print('R2 Score: {}'.format(lr_parsi_score.round(2)))